In [6]:
import sys
import requests
import pandas as pd
from jobspy import scrape_jobs
from datetime import datetime

In [2]:
sys.version

'3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]'

In [3]:
from jobspy import scrape_jobs

df = scrape_jobs(
    site_name=["linkedin", "indeed"],
    search_term="data analyst",
    location="United States",
    results_wanted=10,
    hours_old=72,
    country_indeed="USA"
)

2026-07-10 22:20:26,931 - INFO - JobSpy:Linkedin - finished scraping


In [4]:
df.shape

(20, 34)

In [5]:
df.head()

,id,site,job_url,job_url_direct,title,company,location,date_posted,job_type,salary_source,...,company_addresses,company_num_employees,company_revenue,company_description,skills,experience_range,company_rating,company_reviews_count,vacancy_count,work_from_home_type
0,in-3d91507511787ce2,indeed,https://www.indeed.com/viewjob?jk=3d9150751178...,https://careers.nordstrom.com/data-analyst-3-t...,Data Analyst 3 - Transportation - HYBRID (Seat...,Nordstrom,"Seattle, WA, US",2026-07-10,NaN,direct_data,...,"Seattle, WA","10,000+",more than $10B (USD),"Nordstrom, Inc. is a leading fashion retailer ...",None,None,None,None,None,None
1,in-9be1f40746b0c193,indeed,https://www.indeed.com/viewjob?jk=9be1f40746b0...,https://www.gdit.com/careers/job/94dfbd37b/pro...,Program/Data Analyst Advisor,General Dynamics Information Technology,"Remote, US",2026-07-10,NaN,direct_data,...,"Falls Church, VA","10,000+",more than $10B (USD),We are GDIT. The people supporting some of the...,None,None,None,None,None,None
2,in-dce7a57d29578d54,indeed,https://www.indeed.com/viewjob?jk=dce7a57d2957...,https://deloitteus.avature.net/careers/InviteT...,Clinical Data Analyst,Deloitte,"Arlington, VA, US",2026-07-10,fulltime,direct_data,...,"New York, NY","10,000+",more than $10B (USD),"We’re a leading professional services firm, bu...",None,None,None,None,None,None
3,in-76f635cd70a114f1,indeed,https://www.indeed.com/viewjob?jk=76f635cd70a1...,https://www.paycomdfw.net/v4/ats/web.php/porta...,Revenue Cycle Business Intelligence (BI) Analyst,GI ALLIANCE,"Southlake, TX, US",2026-07-10,fulltime,NaN,...,"Southlake, TX","5,001 to 10,000",$500M to $1B (USD),NaN,None,None,None,None,None,None
4,in-a19af956925dc87d,indeed,https://www.indeed.com/viewjob?jk=a19af956925d...,https://careers-unitypoint.icims.com/jobs/1844...,Compliance Analyst,UnityPoint Health,"Madison, WI, US",2026-07-10,fulltime,NaN,...,NaN,"10,000+",Decline to state,We're proud to be recognized as a Top Place to...,None,None,None,None,None,None


In [7]:
API_KEY = "WYvjQd/FlCzygwHh1hQ9bjEWflR+6GhiiVEFIpvrT+o="
EMAIL   = "a.battini@myemail.indwes.edu"

In [18]:
KEYWORDS = [# Data & Analytics
            "data analyst", "data engineer", "data scientist", "business analyst",
            "business intelligence analyst", "analytics engineer", "quantitative analyst",
            
            # Engineering & Development  
            "software engineer", "cloud engineer", "DevOps engineer",
            "machine learning engineer", "AI engineer",
            
            # IT & Systems
            "IT specialist", "systems analyst", "database administrator",
            "network engineer", "cybersecurity analyst",
            
            # Management & Operations
            "program analyst", "project manager", "operations analyst"]

In [19]:
## USA JOBS

def fetch_usajobs(keyword):
    headers = {
        "Authorization-Key": API_KEY,
        "User-Agent": EMAIL,
        "Host": "data.usajobs.gov"
    }
    r = requests.get(
        "https://data.usajobs.gov/api/search",
        headers=headers,
        params={"Keyword": keyword, "ResultsPerPage": 50}
    )
    items = r.json()["SearchResult"]["SearchResultItems"]
    rows = []
    for item in items:
        d = item["MatchedObjectDescriptor"]
        rows.append({
            "source":           "USAJobs",
            "search_keyword":   keyword,
            "job_id":           d.get("PositionID", ""),
            "title":            d.get("PositionTitle", ""),
            "company":          d.get("OrganizationName", ""),
            "department":       d.get("DepartmentName", ""),
            "location":         ", ".join([l.get("LocationName", "") for l in d.get("PositionLocation", [])]),
            "salary_min":       d.get("PositionRemuneration", [{}])[0].get("MinimumRange", None),
            "salary_max":       d.get("PositionRemuneration", [{}])[0].get("MaximumRange", None),
            "salary_interval":  d.get("PositionRemuneration", [{}])[0].get("RateIntervalCode", None),
            "posted_date":      d.get("PublicationStartDate", "")[:10],
            "close_date":       d.get("ApplicationCloseDate", "")[:10],
            "job_url":          d.get("PositionURI", ""),
            "job_summary":      d.get("UserArea", {}).get("Details", {}).get("JobSummary", "")[:500],
            "work_schedule":    d.get("PositionSchedule", [{}])[0].get("Name", ""),
            "position_type":    d.get("PositionOfferingType", [{}])[0].get("Name", ""),
            "security_clearance": d.get("UserArea", {}).get("Details", {}).get("SecurityClearance", ""),
            "remote":           None,
            "collected_at":     datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
        })
    return rows

In [28]:
headers = {
        "Authorization-Key": API_KEY,
        "User-Agent": EMAIL,
        "Host": "data.usajobs.gov"
    }
r = requests.get(
        "https://data.usajobs.gov/api/search",
        headers=headers,
        params={"Keyword": keyword, "ResultsPerPage": 50})
items = r.json()["SearchResult"]["SearchResultItems"]
items[0]

{'MatchedObjectId': '875687900',
 'MatchedObjectDescriptor': {'PositionID': '26-BPA-51990-13001036-BPA',
  'PositionTitle': 'Market and Pricing Analyst (Operations Research Analyst)',
  'PositionURI': 'https://www.usajobs.gov:443/job/875687900',
  'ApplyURI': ['https://www.usajobs.gov:443/job/875687900'],
  'PositionLocationDisplay': 'Portland, Oregon',
  'PositionLocation': [{'LocationName': 'Portland, Oregon',
    'CountryCode': 'United States',
    'CountrySubDivisionCode': 'Oregon',
    'CityName': 'Portland, Oregon',
    'Longitude': -122.67563,
    'Latitude': 45.511795}],
  'OrganizationName': 'Bonneville Power Administration',
  'DepartmentName': 'Department of Energy',
  'JobCategory': [{'Name': 'Operations Research', 'Code': '1515'}],
  'JobGrade': [{'Code': 'GS'}],
  'PositionSchedule': [{'Name': '', 'Code': '1'}],
  'PositionOfferingType': [{'Name': '', 'Code': '15317'}],
  'QualificationSummary': 'BASIC REQUIREMENTS: Degree: in operations research; or at least 24 semester 

In [30]:
df = scrape_jobs(
            site_name=["linkedin", "indeed"],
            search_term=keyword,
            location="United States",
            results_wanted=5,
            hours_old=24,
            country_indeed="USA")

df.head()

,id,site,job_url,job_url_direct,title,company,location,date_posted,job_type,salary_source,...,company_addresses,company_num_employees,company_revenue,company_description,skills,experience_range,company_rating,company_reviews_count,vacancy_count,work_from_home_type
0,in-566c5831fcb520e0,indeed,https://www.indeed.com/viewjob?jk=566c5831fcb5...,https://www.paycomonline.net/v4/ats/web.php/po...,Direct Support Prof 2 - 1260,Central Counties Services,US,2026-07-10,fulltime,NaN,...,NaN,NaN,NaN,NaN,None,None,None,None,None,None
1,in-b332550ca98bc9b1,indeed,https://www.indeed.com/viewjob?jk=b332550ca98b...,https://ecvz.fa.us2.oraclecloud.com/hcmUI/Cand...,EHR Certified Systems Analyst II,Adventist Health,"Roseville, CA, US",2026-07-10,fulltime,direct_data,...,NaN,NaN,NaN,NaN,None,None,None,None,None,None
2,in-4c983e7a02ede5dc,indeed,https://www.indeed.com/viewjob?jk=4c983e7a02ed...,https://www.paycomonline.net/v4/ats/web.php/po...,Information Technology Support Specialist,NaN,"Colorado Springs, CO, US",2026-07-10,fulltime,direct_data,...,NaN,NaN,NaN,NaN,None,None,None,None,None,None
3,in-05452d517d484970,indeed,https://www.indeed.com/viewjob?jk=05452d517d48...,https://workforcenow.adp.com/mascsr/default/md...,Systems and Marketing Analyst,SUN COMMUNITY FEDERAL CREDIT U,"El Centro, CA, US",2026-07-10,fulltime,direct_data,...,NaN,NaN,NaN,NaN,None,None,None,None,None,None
4,in-d2422b595543b14b,indeed,https://www.indeed.com/viewjob?jk=d2422b595543...,https://jsv3.recruitics.com/redirect?rx_cid=35...,Director of ABA Services,ABS Kids,"Columbia, SC, US",2026-07-09,fulltime,direct_data,...,"Encino, CA","1,001 to 5,000",$100M to $500M (USD),To empower children with autism spectrum disor...,None,None,None,None,None,None


In [20]:
## LINKDIN & INDEED

def fetch_web_jobs(keyword):
    try:
        df = scrape_jobs(
            site_name=["linkedin", "indeed"],
            search_term=keyword,
            location="United States",
            results_wanted=50,
            hours_old=168,
            country_indeed="USA"
        )
        rows = []
        for _, r in df.iterrows():
            rows.append({
                "source":           str(r.get("site", "")).title(),
                "search_keyword":   keyword,
                "job_id":           str(r.get("id", "")),
                "title":            r.get("title", ""),
                "company":          r.get("company", ""),
                "department":       None,
                "location":         r.get("location", ""),
                "salary_min":       r.get("min_amount", None),
                "salary_max":       r.get("max_amount", None),
                "salary_interval":  r.get("interval", None),
                "posted_date":      str(r.get("date_posted", ""))[:10],
                "close_date":       None,
                "job_url":          r.get("job_url", ""),
                "job_summary":      str(r.get("description", ""))[:500],
                "work_schedule":    str(r.get("job_type", "")),
                "position_type":    None,
                "security_clearance": None,
                "remote":           r.get("work_from_home_type", None),
                "collected_at":     datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
            })
        return rows
    except Exception as e:
        print(f"  Error scraping '{keyword}': {e}")
        return []

In [21]:
all_rows = []
seen_ids = set()

for keyword in KEYWORDS:
    
    usa_rows = fetch_usajobs(keyword)
    for row in usa_rows:
        uid = f"usajobs_{row['job_id']}"
        if uid not in seen_ids:
            seen_ids.add(uid)
            all_rows.append(row)
    
   
    web_rows = fetch_web_jobs(keyword)
    for row in web_rows:
        uid = f"{row['source']}_{row['job_id']}"
        if uid not in seen_ids:
            seen_ids.add(uid)
            all_rows.append(row)

df_combined = pd.DataFrame(all_rows)

In [22]:
df_combined.shape

(2124, 19)

In [23]:
df_combined['source'].value_counts()

source
Linkedin    962
Indeed      673
USAJobs     489
Name: count, dtype: int64

In [24]:
import os

In [25]:
os.makedirs("data", exist_ok=True)

date_str = datetime.utcnow().strftime("%Y%m%d")
filename = f"data/jobs_combined_{date_str}.csv"

df_combined.to_csv(filename, index=False)

In [26]:
df_combined.columns

Index(['source', 'search_keyword', 'job_id', 'title', 'company', 'department',
       'location', 'salary_min', 'salary_max', 'salary_interval',
       'posted_date', 'close_date', 'job_url', 'job_summary', 'work_schedule',
       'position_type', 'security_clearance', 'remote', 'collected_at'],
      dtype='object')